# Data Science Task: QS World University Rankings 2025


---
## Section 1: Import Libraries

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


---
## Section 2: Load Dataset

### Q1. Data Loading & Inspection

In [ ]:
data = pd.read_csv("QS_World_University_Rankings_2025_Top_global_universities.csv", encoding="latin-1")


In [ ]:
data.head(5)


In [ ]:
data.shape


In [ ]:
data.info()


In [ ]:
data.isnull().sum()


---
## Section 3: Data Cleaning

### Q2. Data Cleaning

**Data Cleaning Logic:**

- `Overall_Score` and `RANK_2025` are stored as strings (some have `=` suffixes like `15=`). We strip those and convert to numeric.
- Score columns already numeric; rank columns (string) get the same treatment.
- `Overall_Score` has ~902 missing values — these are universities ranked beyond 500 where QS does not publish a score. We impute with the **median** so we keep all rows.
- Other score columns with small gaps are also median-imputed.
- Categorical columns (`STATUS`, `RANK_2024`) with few nulls are filled with `'Unknown'` / forward-fill.


In [ ]:
print('Shape before cleaning:', data.shape)
print(data.isnull().sum())


In [ ]:
df = data.copy()

def strip_rank(val):
    """Remove trailing = or + from rank strings and return int, NaN if unparseable."""
    if pd.isna(val):
        return np.nan
    cleaned = str(val).strip().rstrip('=+').split('-')[0]
    try:
        return int(cleaned)
    except ValueError:
        return np.nan

rank_cols = [c for c in df.columns if 'Rank' in c or c.startswith('RANK')]
for col in rank_cols:
    df[col] = df[col].apply(strip_rank)

df['Overall_Score'] = pd.to_numeric(
    df['Overall_Score'].astype(str).str.rstrip('=+'), errors='coerce'
)


In [ ]:
score_cols = [c for c in df.columns if 'Score' in c]
for col in score_cols:
    med = df[col].median()
    df[col] = df[col].fillna(med)

df['STATUS'] = df['STATUS'].fillna('Unknown')
df['RANK_2024'] = df['RANK_2024'].fillna(method='ffill')

print('Shape after cleaning:', df.shape)
print(df.isnull().sum())


In [ ]:
df.head(5)


### Q3. Feature Engineering (Using NumPy)

**Score_Category rules:**
- Elite → Overall_Score ≥ 90
- High → 75–89
- Medium → 50–74
- Low → < 50


In [ ]:
s = df['Overall_Score'].values
cats = np.where(s >= 90, 'Elite',
        np.where(s >= 75, 'High',
        np.where(s >= 50, 'Medium', 'Low')))
df['Score_Category'] = cats
df[['Institution_Name', 'Overall_Score', 'Score_Category']].head(10)


In [ ]:
df['Score_Category'].value_counts()


---
## Section 4: EDA (Exploratory Data Analysis)

### Q4. Top Universities Analysis

In [ ]:
top10 = df.nsmallest(10, 'RANK_2025')[['Institution_Name', 'RANK_2025', 'Overall_Score', 'Location']]
top10


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette('Blues_r', 10)
bars = ax.barh(top10['Institution_Name'][::-1], top10['Overall_Score'][::-1], color=colors)
ax.set_xlabel('Overall Score', fontsize=12)
ax.set_title('Top 10 Universities by QS Rank 2025', fontsize=14, fontweight='bold')
for bar, score in zip(bars, top10['Overall_Score'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()


**Insights:**

1. MIT holds the top spot with a perfect score of 100, maintaining its #1 position from 2024.
2. UK universities (Oxford, Cambridge, Imperial) dominate the top 5 alongside US institutions.
3. The score gap between rank 1 (100) and rank 10 (~90.9) is only ~9 points — the elite tier is tightly clustered.


### Q5. Country-wise Dominance

In [ ]:
country_counts = df['Location'].value_counts().reset_index()
country_counts.columns = ['Country', 'University_Count']
top10_countries = country_counts.head(10)
top10_countries


In [ ]:
fig = px.bar(
    top10_countries,
    x='Country', y='University_Count',
    color='University_Count',
    color_continuous_scale='Blues',
    title='Top 10 Countries by Number of Ranked Universities (QS 2025)',
    labels={'University_Count': 'Number of Universities'},
    text='University_Count'
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_tickangle=-30)
fig.show()


**Interpretation:**

The United States leads by a wide margin, followed by the UK. Asian countries like China, Japan, and South Korea are strongly represented. Australia punches above its weight given its population size.


### Q6. Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df['Overall_Score'], bins=40, kde=True, color='steelblue', ax=ax)
ax.set_title('Distribution of Overall Scores (QS 2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Overall Score')
ax.set_ylabel('Count')
ax.axvline(df['Overall_Score'].median(), color='red', linestyle='--', label=f"Median: {df['Overall_Score'].median():.1f}")
ax.legend()
plt.tight_layout()
plt.show()


**Analysis:**

- Distribution type: **Right-skewed**
- The majority of universities cluster in the lower score range (below 50), with a long tail stretching toward 100. Only a handful of elite institutions score above 90.


### Q7. Correlation Analysis

In [ ]:
num_cols = [
    'Academic_Reputation_Score', 'Employer_Reputation_Score',
    'Faculty_Student_Score', 'Citations_per_Faculty_Score',
    'International_Faculty_Score', 'International_Students_Score',
    'Employment_Outcomes_Score', 'Sustainability_Score', 'Overall_Score'
]
corr = df[num_cols].corr()
corr


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5, ax=ax
)
ax.set_title('Correlation Heatmap of Score Columns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Strongest Relationships:**

- `Academic_Reputation_Score` and `Employer_Reputation_Score` show a strong positive correlation (~0.85).
- `Overall_Score` correlates most strongly with `Academic_Reputation_Score` and `Employer_Reputation_Score`.
- `Citations_per_Faculty_Score` has a moderate correlation with `Overall_Score`.


### Q8. Region-wise Performance

In [ ]:
region_avg = df.groupby('Region')['Overall_Score'].mean().sort_values(ascending=False).reset_index()
region_avg.columns = ['Region', 'Avg_Score']
region_avg


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(region_avg['Region'], region_avg['Avg_Score'],
              color=sns.color_palette('Set2', len(region_avg)))
ax.set_title('Average Overall Score by Region (QS 2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Region')
ax.set_ylabel('Average Score')
for bar, val in zip(bars, region_avg['Avg_Score']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', fontsize=9)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


**Interpretation:**

The Americas and Europe lead in average overall score, driven by elite US and UK universities. Asia is competitive but its average is pulled down by the large number of lower-ranked institutions. Africa and Oceania trail behind.


---
## Section 5: Analytical Use Cases

### Q9. Rank vs Score Relationship

In [ ]:
scatter_df = df[['Institution_Name', 'RANK_2025', 'Overall_Score', 'Region']].dropna()
fig = px.scatter(
    scatter_df, x='RANK_2025', y='Overall_Score',
    color='Region', hover_name='Institution_Name',
    trendline='ols',
    title='Rank vs Overall Score (QS 2025)',
    labels={'RANK_2025': 'Rank 2025', 'Overall_Score': 'Overall Score'}
)
fig.show()


**Comment on Relationship:**

There is a strong negative relationship between rank and score — as rank number increases (lower position), the overall score drops sharply. The relationship is non-linear: scores fall steeply in the top 100 and then flatten out, meaning the difference between rank 500 and rank 1000 is much smaller than between rank 1 and rank 100.


### Q10. Top University per Country

In [ ]:
def get_top_uni(group):
    """Return the row with the lowest (best) rank in each country group."""
    return group.loc[group['RANK_2025'].idxmin()]

top_per_country = (
    df.groupby('Location', group_keys=False)
      .apply(get_top_uni)
      [['Location', 'Institution_Name', 'RANK_2025', 'Overall_Score']]
      .sort_values('RANK_2025')
      .reset_index(drop=True)
)
top_per_country.head(20)


### Q11. Outlier Detection

In [ ]:
col = 'Overall_Score'
q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = df[(df[col] < lower) | (df[col] > upper)]
print(f'IQR range: [{lower:.2f}, {upper:.2f}]')
print(f'Number of outliers: {len(outliers)}')
outliers[['Institution_Name', 'Overall_Score', 'RANK_2025']]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(x=df['Overall_Score'], color='lightcoral', ax=ax)
ax.set_title('Boxplot of Overall Score — Outlier Detection', fontsize=13, fontweight='bold')
ax.set_xlabel('Overall Score')
plt.tight_layout()
plt.show()


**Number of Outliers Found:**

The IQR method flags the top-scoring universities (roughly those scoring above ~85) as statistical outliers. This is expected — these are genuinely exceptional institutions that sit far above the bulk of the ranked list. They are not data errors; they represent the true elite tier.


### Q12. Score Category Analysis

In [ ]:
cat_counts = df['Score_Category'].value_counts().reset_index()
cat_counts.columns = ['Category', 'Count']
cat_counts


In [ ]:
order = ['Elite', 'High', 'Medium', 'Low']
cat_counts['Category'] = pd.Categorical(cat_counts['Category'], categories=order, ordered=True)
cat_counts = cat_counts.sort_values('Category')

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
bars = ax.bar(cat_counts['Category'], cat_counts['Count'], color=colors)
ax.set_title('Universities by Score Category (QS 2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Score Category')
ax.set_ylabel('Number of Universities')
for bar, val in zip(bars, cat_counts['Count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', fontsize=10)
plt.tight_layout()
plt.show()


**Interpretation:**

The vast majority of ranked universities fall in the **Low** category (score < 50), which reflects how competitive the global rankings are — most institutions that make the list are still far from the elite tier. The **Elite** category (score ≥ 90) contains only a tiny fraction, confirming that truly world-class universities are rare.
